# HOMEWORK 15

In this homework we will be working with the Fashion MNIST dataset. You will be given a classifier which suffers from considerable overfitting. Your objective will be to employ regularization techniques to mitigate the overfitting problem.

Let's start with the usual imports.

In [ ]:
import time
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

import numpy as np
import torch
import keras
from keras import Model, regularizers
from keras.layers import (
    Activation, BatchNormalization, Conv2D, Dense, 
    Dropout, Flatten, Input, MaxPooling2D,  SpatialDropout2D, GlobalAveragePooling2D
)
from sklearn.model_selection import train_test_split

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

# Set the seed for reproducibility (python, numpy and the backend)
keras.utils.set_random_seed(1234)

# A lot of training in this hw, so I'll use my GPU to speed things up
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

### Dataset

The Fashion MNIST dataset [link](https://github.com/zalandoresearch/fashion-mnist) was built by Zalando Research and consists of monochrome images of different types of clothing, namely:
* 0	T-shirt/top
* 1	Trouser
* 2	Pullover
* 3	Dress
* 4	Coat
* 5	Sandal
* 6	Shirt
* 7	Sneaker
* 8	Bag
* 9	Ankle boot

It is also one of the Keras built-in datasets. Let's load the images and quickly inspect it. We keep the 10,000 test images aside for the final evaluation and hold out 6,000 of the training images as the validation set, which is used to monitor the training and to take decisions.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
num_classes = len(class_names)

# Normalization
x_train = x_train.astype('float32')/255
x_test = x_test.astype('float32')/255

# Hold out a (class-balanced) validation set
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=6000, stratify=y_train, random_state=0)

print('Train set:      ', len(y_train), 'samples')
print('Validation set: ', len(y_val), 'samples')
print('Test set:       ', len(y_test), 'samples')
print('Sample dims:    ', x_train.shape)

Let's visualise some random samples.

In [ ]:
cnt = 1
for r in range(3):
    for c in range(6):
        idx = np.random.randint(len(x_train))
        plt.subplot(3, 6, cnt)
        plt.imshow(x_train[idx, ...], cmap='gray')
        plt.title(class_names[y_train[idx]])
        cnt = cnt + 1

### Building the Classifier

We are now going to build the baseline classifier that you will use throughout this homework.

In [ ]:
inputs = Input(shape=(28, 28, 1))
net = Conv2D(32, kernel_size=(3, 3), activation="relu", padding='same')(inputs)
net = Flatten()(net)
net = Dense(128, activation="relu")(net)
outputs = Dense(10, activation="softmax")(net)

model = Model(inputs, outputs)
model.summary()

# Move model to GPU
model.to("cuda")

# Verify
print(next(model.parameters()).device)  # cuda:0

In [ ]:
epochs = 50
batch_size = 64

model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(x_val, y_val), verbose=2)

In [ ]:
def plot_history(history):
    h = history.history
    epochs = range(len(h['loss']))

    plt.subplot(121), plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
    plt.legend(['Train', 'Validation'])
    plt.subplot(122), plt.plot(epochs, h['accuracy'], '.-',
                               epochs, h['val_accuracy'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Accuracy')
    plt.legend(['Train', 'Validation'])

    print('Train Acc     ', h['accuracy'][-1])
    print('Validation Acc', h['val_accuracy'][-1])

plot_history(history)

As you can see, the classifier suffers from massive overfitting. The training accuracy is close to 1 (0.998), while the validation accuracy is only about 90%. The validation loss reaches its minimum after just a few epochs (the validation accuracy peaks at about 91.5% in epoch 6) and then it keeps growing steadily, which is the typical sign of overfitting.

### Combat the Overfitting!

Now it is your turn. Use the classifier as a baseline, include some regularization techniques and try to improve the classification performance. You can try any techniques you might see fit, e.g.,
* Dropout (done)
* Batch normalization (done)
* Weight regularization (done)
* Data augmentation
* Early stopping (done)
* Pooling (done)
* Reducing the number of parameters (the size of the network) (done)
* ...

+ SpatialDropout2D (done)
+ ReduceLROnPlateau (done)

There are two objectives you shall fulfill in order to successfully complete this homework:
* The validation accuracy shall be above 91%
* Your network (with all the regularizations applied) shall **not** be larger than the baseline

Use the validation set for all your decisions. The test set is meant only for the final evaluation of your best network.

**Note:** `fit` continues training the model it is called on. Whenever you start a new experiment (even if you change only the training, e.g. the augmentation), run the cell that builds the model again to get a freshly initialized one.

In [ ]:
# Modify the baseline classifier in order to reduce the overfitting and make the performance more robust

inputs = Input(shape=(28, 28, 1))
net = Conv2D(32, kernel_size=(3, 3), padding="same", kernel_regularizer=regularizers.L2(1e-4))(inputs)
net = BatchNormalization(momentum=0.9)(net)    
net = Activation("relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = SpatialDropout2D(0.1)(net)
net = Conv2D(64, kernel_size=(3, 3), padding="same", kernel_regularizer=regularizers.L2(1e-4))(net)
net = BatchNormalization(momentum=0.9)(net)    
net = Activation("relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = SpatialDropout2D(0.15)(net)
net = Conv2D(128, kernel_size=(3, 3), padding="same", kernel_regularizer=regularizers.L2(1e-4))(net)
net = BatchNormalization(momentum=0.9)(net)    
net = Activation("relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = SpatialDropout2D(0.2)(net)
net = GlobalAveragePooling2D()(net)
net = Dense(128, activation="relu", kernel_regularizer=regularizers.L2(1e-4))(net)
net = Dropout(rate=0.3)(net)

outputs = Dense(10, activation="softmax")(net)

model = Model(inputs, outputs)
model.summary()

# Move model to GPU
model.to("cuda")

# Verify
print(next(model.parameters()).device)  # cuda:0

In [ ]:
# Train the network
epochs = 100 # more epochs, no account for dropout
batch_size = 128
callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    )
]

model.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.Adam(learning_rate=3e-4), metrics=["accuracy"])
history = model.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(x_val, y_val), callbacks=callbacks, shuffle=False, verbose=2)

In [ ]:
# Show the results

def plot_history(history):
    h = history.history
    epochs = range(len(h['loss']))

    plt.subplot(121), plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
    plt.legend(['Train', 'Validation'])
    plt.subplot(122), plt.plot(epochs, h['accuracy'], '.-',
                               epochs, h['val_accuracy'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Accuracy')
    plt.legend(['Train', 'Validation'])

    print('Train Acc     ', h['accuracy'][-1])
    print('Validation Acc', h['val_accuracy'][-1])

plot_history(history)

In [ ]:
test_loss, test_acc = model.evaluate(
    x_test, y_test,
    batch_size=128,
    verbose=0
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

### Questions

* What have you done in order to improve the performance?

Everything except data augmentation. I was rusty with torch DataLoaders API and I wasn't comfortable with generators either, so I concentrated on architecture and regularization instead. I had 3 version of the model (the last two graphs are in the homework folder, I didn't save v1, unforturanely).

I did the following:
- Pushed feature selection from dense layer to conv layers. I've added Conv layers with maxpooling and used global pooling instead of flattenning to reduce params. The resulting architecture is 30x smaller than the original! 
- Regularized the model with l2 weight reg at each layer. Added BatchNorm before activations. Added SpatialDrop out for conv layers and usual one for dense layer (that's why val acc is so good at first). Also I've added early stopping to stop trainging when thing go South. I've also increased batch size to 128 which results into better BN params and faster training.
- The resulting model is v1 and it had the problem of jittery gradient. I decided to specify lr 1e-4 for Adam, gave lower momentum to BN (0.9 instead of 0.99) for them to better adapt to changes during training. Also, I've added reduce lr on plateau callback.
- The resulting model is v2 (image of training in hw folder). Even with 30x less params the model was still overfitting! So I have decided to push dense dropout (0.2 -> 0.3), and add spacial dropouts for each conv layer instead of just the last one in v1-2.
- The resulting model is v3 and it's architecture is in this notebook. I achieved val acc of 0.93 and train-val diff of only 0.9! The test acc is 0.93!

* Have you tried configurations that did not work out?

Some of it I've covered above but, in general, my implementation of dataloaders made training extra slow (epoch in minutes instead of seconds on GPU). Definitely, there was some bug there that I couldn't find, so after an hour or so I gave up and stick to modding architecture.